# Lesson 7: Measuring, attributing, and capping LLM spend

*Module 4 · about 12 minutes · API key required (uses the cheapest tier, so it costs well under a cent)*

Lessons 1–6 were about spending less. This one is about knowing what you spend, who spent it, and stopping it when it runs away. Without that, the savings from the earlier lessons are hard to prove and easy to lose: nobody can show the bill went down, nobody notices when it creeps back up, and nothing stops an agent stuck in a loop at 3 a.m.

FinOps practitioners describe the order of work as **Track → Attribute → Control → Optimize**. Most teams start with Optimize because it's the interesting part, and then find they can't measure whether it worked. We'll go through the first three here. Everything runs inside the notebook, with no gateway, no Docker, and no observability vendor.

By the end you should be able to:

1. Tag every model call with the fields you need for chargeback and cost-per-task reporting.
2. Build the basic spend views and a tag-coverage check.
3. Set a hard budget per team, prove that it fires, and show that one team's overrun doesn't block another team.
4. Stop a runaway agent with independent limits on steps, tokens, time, and money.
5. Report cost per completed task rather than cost per call.


### How these notebooks work

Run the cells in order, top to bottom. Before each code cell there's a short explanation of what it does and what to look at in the output. After the important ones there's a note on how to read what you got. Your numbers won't match mine exactly, because models are non-deterministic and prices change, so the notes describe what to look for rather than quoting fixed values.

A few conventions:

- **In class:** notes are cues for when we run this together. If you're working alone, just read them as a prompt to stop and think.
- Every notebook that spends money ends with a **ledger**: one row per API call and the total you spent.
- The **Check yourself** questions at the end have answers hidden under a click. Try them before you look.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, vendor_tokens, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
LIVE = cfg.live


  Provider : anthropic
  floor    : claude-haiku-4-5
  mid      : claude-sonnet-5
  frontier : claude-opus-5
  Cache    : explicit cache_control; read/write are separate buckets.
Switch with LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env
  Rate card: verified 5 Sep 2026 — re-check before presenting.


That cell reads your `.env`, picks OpenAI or Anthropic depending on which key it finds, and prints the three model tiers the notebook will use (floor, mid, frontier).

If the banner names a provider, the live cells will make real calls. Every lesson costs cents, not dollars. If it says `offline`, all the arithmetic still runs, but cells that need a model's answer print a placeholder and tell you they can't draw a conclusion. You can read an offline run, but it's no substitute for a live one in the caching, compression, and routing lessons.

To switch vendors, set `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and run the cell again.


---
## 1. A wrapper that records every call

Everything in this lesson hangs off one function, `instrumented_call()`. It does three things around each model call:

1. **Checks the budget before calling.** If the team is already over its limit, the call is refused. Note that the check runs *before* the call, against spend so far, so the call that crosses the line is allowed and the next one is blocked. Budgets can therefore overshoot by one call. Real gateways behave the same way, which is one reason to keep `max_tokens` bounded.
2. **Makes the call** through `complete()`.
3. **Records a span**, a structured record of one operation, in the shape used by OpenTelemetry (OTel), the open standard most observability tools accept.

The span has two kinds of fields:

- **Standard GenAI fields** from the OTel GenAI semantic conventions: `gen_ai.operation.name`, `gen_ai.provider.name`, `gen_ai.request.model`, `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens`, `gen_ai.usage.cache_read.input_tokens`, `gen_ai.usage.cache_creation.input_tokens`. The conventions say `input_tokens` should *include* the cached tokens, so for Anthropic we add the three input buckets back together. As of mid-2026 these conventions are still marked *Development* rather than stable, so pin the version you follow.
- **Your own attribution fields**: `business_unit`, `use_case_id`, `environment`, `user_id`, `session_id`, and so on. These aren't part of any standard. They're what makes the spend answerable to "whose is this?".


In [2]:
import time, uuid, json
from collections import defaultdict

TRACES = []
BUDGETS = {}
SPEND = defaultdict(float)

class BudgetExceeded(Exception):
    pass

class CircuitBreakerTripped(Exception):
    pass

def instrumented_call(prompt, *, model=None, max_tokens=80,
                      business_unit=None, use_case_id=None, environment="prod",
                      workload_type="inference", user_id=None,
                      session_id=None, cost_tier=2, budget_key=None):
    """One model call: budget-checked, tagged, and recorded as an OTel-style span."""
    model = model or MODELS.floor
    budget_key = budget_key or business_unit or "unallocated"

    limit = BUDGETS.get(budget_key)
    if limit is not None and SPEND[budget_key] >= limit:           # check BEFORE spending
        raise BudgetExceeded(
            f"budget_exceeded: key={budget_key} spend={usd(SPEND[budget_key])} limit={usd(limit)}"
        )

    t0 = time.time()
    r = complete(prompt, model=model, max_tokens=max_tokens)
    SPEND[budget_key] += r.usd

    span = {
        # standard OTel GenAI attributes
        "gen_ai.operation.name": "chat",
        "gen_ai.provider.name": cfg.provider,
        "gen_ai.request.model": r.model,
        "gen_ai.usage.input_tokens": r.fresh_input + r.cache_read + r.cache_write,
        "gen_ai.usage.output_tokens": r.output_tokens,
        "gen_ai.usage.cache_read.input_tokens": r.cache_read,
        "gen_ai.usage.cache_creation.input_tokens": r.cache_write,
        # your own attribution and cost fields
        "business_unit": business_unit,
        "use_case_id": use_case_id,
        "environment": environment,
        "workload_type": workload_type,
        "user_id": user_id,
        "session_id": session_id or str(uuid.uuid4())[:8],
        "cost_tier": cost_tier,
        "duration_s": round(time.time() - t0, 3),
        "usd": r.usd,
    }
    TRACES.append(span)
    log_call(f"{business_unit}/{use_case_id}", r.model,
             inp=r.fresh_input, out=r.output_tokens,
             cache_w=r.cache_write, cache_r=r.cache_read)
    return r.text, span

print("instrumented client ready, using", MODELS.floor)


instrumented client ready, using claude-haiku-4-5


---
## 2. Some traffic to measure

Ten short calls from five use cases across three business units: customer support and sentiment classification for customer operations, code review for engineering, and question answering and summarisation for sales. Each call is tagged.

We add two more calls from a "legacy batch script" that nobody tagged. That's what real traffic looks like: some of it always comes from a script someone wrote before the tagging rules existed.


In [3]:
WORKLOAD = [
    ("support",        "uc-101", "customer-ops", "Summarise: shipment NW-1042 delayed 51 hours, customer claim $410."),
    ("support",        "uc-101", "customer-ops", "Summarise: shipment NW-1043 delayed 12 hours, no claim filed."),
    ("classification", "uc-102", "customer-ops", 'Classify sentiment, one word: "Third delay this month. Unacceptable."'),
    ("classification", "uc-102", "customer-ops", 'Classify sentiment, one word: "Arrived early, great service."'),
    ("code_review",    "uc-201", "engineering",  "In one sentence: what is wrong with `for i in range(len(x)): print(x[i])`?"),
    ("code_review",    "uc-201", "engineering",  "In one sentence: why prefer a context manager over manual file close?"),
    ("rag_answer",     "uc-301", "sales-mktg",   "One sentence: what is the business case for prompt caching?"),
    ("rag_answer",     "uc-301", "sales-mktg",   "One sentence: when does self-hosting an LLM beat an API?"),
    ("summarisation",  "uc-302", "sales-mktg",   "One sentence: summarise the concept of cost per completed task."),
    ("summarisation",  "uc-302", "sales-mktg",   "One sentence: summarise why output tokens cost more than input."),
]
USERS = {"customer-ops": "u-104", "engineering": "u-221", "sales-mktg": "u-317"}

for name, uc, bu, prompt in WORKLOAD:
    _, span = instrumented_call(prompt, business_unit=bu, use_case_id=uc, user_id=USERS[bu])

# The legacy script: no business unit, no use case, no user.
for prompt in ["Summarise: nightly batch of 40 delivery exceptions.",
               "Summarise: nightly batch of 12 address corrections."]:
    instrumented_call(prompt)

print(f"\n{len(TRACES)} spans captured")


customer-ops/uc-101                           $0.000200   in=30      out=34     cw=0       cr=0       


customer-ops/uc-101                           $0.000229   in=29      out=40     cw=0       cr=0       


customer-ops/uc-102                           $0.000051   in=26      out=5      cw=0       cr=0       


customer-ops/uc-102                           $0.000047   in=22      out=5      cw=0       cr=0       


engineering/uc-201                            $0.000204   in=34      out=34     cw=0       cr=0       


...

sales-mktg/uc-301                             $0.000263   in=23      out=48     cw=0       cr=0       


sales-mktg/uc-302                             $0.000195   in=20      out=35     cw=0       cr=0       


sales-mktg/uc-302                             $0.000155   in=20      out=27     cw=0       cr=0       


None/None                                     $0.000421   in=21      out=80     cw=0       cr=0       


None/None                                     $0.000131   in=21      out=22     cw=0       cr=0       

12 spans captured


---
## 3. Track and attribute: the basic views

Three views on the same spans:

1. **Spend by business unit.** This is the *showback* view (reporting to each team what it spent) and later the *chargeback* view (actually billing it back to their budget). Showback usually comes first. It's much easier to agree on who pays once everyone can see the numbers.
2. **Spend by use case.** This tells you where to point Lessons 1–6: the top two or three use cases usually account for most of the spend.
3. **Tag coverage.** The share of spans that have each required tag. A common target is 95% or more within 30 days of starting. Anything untagged ends up in an "unallocated" bucket that nobody owns, and nobody works to reduce.


In [4]:
df = pd.DataFrame(TRACES)
df["business_unit"] = df.business_unit.fillna("(untagged)")

print("SPEND BY BUSINESS UNIT")
show(df.groupby("business_unit").agg(
        calls=("usd", "size"), usd=("usd", "sum"),
        in_tok=("gen_ai.usage.input_tokens", "sum"),
        out_tok=("gen_ai.usage.output_tokens", "sum"))
     .sort_values("usd", ascending=False)
     .style.format({"usd": "${:,.6f}"}))

print("\nSPEND BY USE CASE")
top = df.groupby(df.use_case_id.fillna("(untagged)")).usd.sum().sort_values(ascending=False)
show(top.to_frame().style.format({"usd": "${:,.6f}"}))

print("\nTAG COVERAGE (share of spans with each tag; target 95%+)")
raw = pd.DataFrame(TRACES)
for tag in ["business_unit", "use_case_id", "user_id", "environment", "session_id",
            "gen_ai.request.model", "cost_tier"]:
    cov = raw[tag].notna().mean()
    print(f"  {tag:<24} {cov:>6.0%}{'   <-- below target' if cov < 0.95 else ''}")
unalloc = raw[raw.business_unit.isna()].usd.sum()
print(f"\nUnallocated spend: {usd(unalloc)} ({unalloc / raw.usd.sum():.0%} of the total)")


SPEND BY BUSINESS UNIT


,calls,usd,in_tok,out_tok
business_unit,,,,
sales-mktg,4,$0.000838,83,151
(untagged),2,$0.000552,42,102
customer-ops,4,$0.000527,107,84
engineering,2,$0.000330,55,55



SPEND BY USE CASE


,usd
use_case_id,
(untagged),$0.000552
uc-301,$0.000488
uc-101,$0.000429
uc-302,$0.000350
uc-201,$0.000330
uc-102,$0.000098



TAG COVERAGE (share of spans with each tag; target 95%+)
  business_unit               83%   <-- below target
  use_case_id                 83%   <-- below target
  user_id                     83%   <-- below target
  environment                100%
  session_id                 100%
  gen_ai.request.model       100%
  cost_tier                  100%

Unallocated spend: $0.000552 (25% of the total)


The model name, environment, and session ID are at 100% because the wrapper fills them in automatically. The business tags aren't, because only the caller knows them. That's the practical lesson: **make the wrapper (or gateway) require the attribution tags**, and reject or flag untagged calls, rather than hoping everyone remembers. You can't go back and tag old calls afterwards. Once the context is gone, it's gone.


---
## 4. Control: a hard budget, and proof that it fires

A budget that has never been seen to fire isn't really a control. It's an assumption. So we'll set one and trigger it.

We give customer-ops a budget just a little above what it has already spent, then send eight more calls. Each call prints the spend as a percentage of the budget: an alert at 80%, and a block once the budget is used up. Engineering gets a comfortable \$5.

Some context on why this matters: in 2026 Uber reported using up its entire annual AI coding-tools budget in about four months, across roughly 5,000 engineers, and then introduced a cap of \$1,500 per employee per tool per month. A cap introduced *after* the money is gone is a reaction. This cell is the version you build in advance.

In production this check lives in a gateway (LiteLLM, Portkey, Kong and similar) rather than in application code. That way no one can bypass it, and it persists spend across processes: LiteLLM's documentation is explicit that budgets are enforced against spend stored in its database, so a gateway with no database can't enforce one at all. When a LiteLLM budget is exceeded, requests fail with a `budget_exceeded` error (HTTP 429).


In [5]:
BUDGETS["customer-ops"] = SPEND["customer-ops"] + 0.00015   # a little headroom, so it trips quickly
BUDGETS["engineering"]  = 5.00                              # plenty of room

print(f"customer-ops budget: {usd(BUDGETS['customer-ops'])}  (already spent {usd(SPEND['customer-ops'])})")
print(f"engineering budget:  {usd(BUDGETS['engineering'])}\n")

blocked = 0
for i in range(8):
    try:
        instrumented_call(f"Reply with the single word OK. ({i})",
                          business_unit="customer-ops", use_case_id="uc-101", user_id="u-104")
        pct = SPEND["customer-ops"] / BUDGETS["customer-ops"]
        status = "OVER BUDGET (next call will be blocked)" if pct >= 1 else ("ALERT: past 80%" if pct >= 0.8 else "ok")
        print(f"  call {i}: allowed   spend={usd(SPEND['customer-ops'])} ({pct:.0%} of budget)  {status}")
    except BudgetExceeded as e:
        blocked += 1
        print(f"  call {i}: BLOCKED   {e}")

print(f"\n{blocked} of 8 calls blocked.")


customer-ops budget: $0.000677  (already spent $0.000527)
engineering budget:  $5.00



customer-ops/uc-101                           $0.000037   in=17      out=4      cw=0       cr=0       
  call 0: allowed   spend=$0.000564 (83% of budget)  ALERT: past 80%


customer-ops/uc-101                           $0.000037   in=17      out=4      cw=0       cr=0       
  call 1: allowed   spend=$0.000601 (89% of budget)  ALERT: past 80%


customer-ops/uc-101                           $0.000037   in=17      out=4      cw=0       cr=0       
  call 2: allowed   spend=$0.000638 (94% of budget)  ALERT: past 80%


customer-ops/uc-101                           $0.000037   in=17      out=4      cw=0       cr=0       
  call 3: allowed   spend=$0.000675 (100% of budget)  ALERT: past 80%


customer-ops/uc-101                           $0.000037   in=17      out=4      cw=0       cr=0       
  call 4: allowed   spend=$0.000712 (105% of budget)  OVER BUDGET (next call will be blocked)
  call 5: BLOCKED   budget_exceeded: key=customer-ops spend=$0.000712 limit=$0.000677
  call 6: BLOCKED   budget_exceeded: key=customer-ops spend=$0.000712 limit=$0.000677
  call 7: BLOCKED   budget_exceeded: key=customer-ops spend=$0.000712 limit=$0.000677

3 of 8 calls blocked.


Notice the overshoot described in section 1: the call that crossed the limit went through, and the block applied from the next call on.

### One team's overrun shouldn't stop everyone else

Customer-ops is now blocked. Engineering should be unaffected, because it has its own budget key. With a single shared API key for the whole company, one team's runaway job would block everybody.


In [6]:
_, s = instrumented_call("Reply with the single word OK.",
                         business_unit="engineering", use_case_id="uc-201", user_id="u-221")
print(f"\nengineering call went through: spend {usd(SPEND['engineering'])} of {usd(BUDGETS['engineering'])}")


engineering/uc-201                            $0.000034   in=14      out=4      cw=0       cr=0       

engineering call went through: spend $0.000364 of $5.00


---
## 5. Circuit breakers for agents

Budgets are monthly or daily. An agent stuck in a loop can do a lot of damage well inside a monthly budget, so agents also need **per-run limits** that stop a single task.

The cell below runs a deliberately broken agent: its goal says "never conclude", and every step appends more text to its context, the way a real agent accumulates tool results. It has four independent ceilings:

| Limit | Catches |
|---|---|
| steps | a loop that keeps going, however cheap each step is |
| tokens | context that grows out of control |
| wall-clock seconds | a slow tool or a hung call that never returns |
| dollars | anything the other three miss |

You need more than one because each catches a different failure. A token limit won't notice a loop stuck waiting on a slow tool. A time limit won't notice a fast loop that spends a lot per second. Whichever limit trips first stops the run.


In [7]:
BREAKERS = dict(max_steps=6, max_tokens=4000, max_seconds=25, max_usd=0.02)

def run_agent(goal, breakers=BREAKERS):
    sid = str(uuid.uuid4())[:8]
    t0, steps, tokens, spend = time.time(), 0, 0, 0.0
    ctx = goal
    while True:
        steps += 1
        if steps > breakers["max_steps"]:
            raise CircuitBreakerTripped(f"STEP LIMIT: {steps - 1} steps (session {sid})")
        if tokens > breakers["max_tokens"]:
            raise CircuitBreakerTripped(f"TOKEN LIMIT: {tokens:,} tokens (session {sid})")
        if time.time() - t0 > breakers["max_seconds"]:
            raise CircuitBreakerTripped(f"TIME LIMIT: {time.time() - t0:.0f}s (session {sid})")
        if spend > breakers["max_usd"]:
            raise CircuitBreakerTripped(f"SPEND LIMIT: {usd(spend)} (session {sid})")

        _, s = instrumented_call(
            ctx + "\n\nYou have not finished. Restate the task in one sentence and stop.",
            business_unit="engineering", use_case_id="uc-999", user_id="u-221",
            session_id=sid, max_tokens=60)
        tokens += s["gen_ai.usage.input_tokens"] + s["gen_ai.usage.output_tokens"]
        spend += s["usd"]
        ctx += " " + ("CONTEXT " * 40)            # the context grows every step, like a real agent's
        print(f"  step {steps}: tokens={tokens:>6,}  spend={usd(spend)}  elapsed={time.time() - t0:.0f}s")

try:
    run_agent("Determine the optimal refund policy. Never conclude.")
except CircuitBreakerTripped as e:
    print(f"\nSTOPPED -> {e}")
except BudgetExceeded as e:
    print(f"\nBLOCKED BY BUDGET -> {e}")


engineering/uc-999                            $0.000105   in=35      out=14     cw=0       cr=0       
  step 1: tokens=    49  spend=$0.000105  elapsed=1s


engineering/uc-999                            $0.000281   in=116     out=33     cw=0       cr=0       
  step 2: tokens=   198  spend=$0.000386  elapsed=2s


engineering/uc-999                            $0.000312   in=197     out=23     cw=0       cr=0       
  step 3: tokens=   418  spend=$0.000698  elapsed=3s


engineering/uc-999                            $0.000433   in=278     out=31     cw=0       cr=0       
  step 4: tokens=   727  spend=$0.001131  elapsed=4s


engineering/uc-999                            $0.000469   in=359     out=22     cw=0       cr=0       
  step 5: tokens= 1,108  spend=$0.001600  elapsed=4s


engineering/uc-999                            $0.000580   in=440     out=28     cw=0       cr=0       
  step 6: tokens= 1,576  spend=$0.002180  elapsed=7s

STOPPED -> STEP LIMIT: 6 steps (session 6b3b2b69)


Try tightening one limit at a time (for example, `max_tokens=1500`) and re-running, to see a different breaker trip first. Also notice that a forecast would never have caught this. Forecasts describe expected behaviour. Limits are what stop the unexpected.


---
## 6. Cost per completed task

Cost per call is easy to compute and misleading for agents, because one user request can make many calls. The looping agent above made six. What you want to report is **cost per completed task**, and that needs a way to group calls into tasks. That's what `session_id` is for.

Below we group spans by session, compute the average cost per task, and then divide by a success rate to get **cost per successful task**, as in Lesson 6. The 92% used here is an assumed figure for illustration. In a real system it comes from your evaluation or from production outcome data, never from a guess.

We also flag sessions that cost far more than the rest. That's a simple way to spot runaway agents after the fact.


In [8]:
df = pd.DataFrame(TRACES)
by_session = (df.groupby("session_id")
                .agg(calls=("usd", "size"), usd=("usd", "sum"),
                     tokens=("gen_ai.usage.input_tokens", "sum"),
                     use_case=("use_case_id", "first"))
                .sort_values("usd", ascending=False))
show(by_session.head(5).style.format({"usd": "${:,.6f}"}))

SUCCESS_RATE = 0.92   # assumed for this demo; in practice, measured by your eval or outcome data
per_task = df.usd.sum() / df.session_id.nunique()
print(f"\ncost per call                 {usd(df.usd.mean())}")
print(f"cost per task (per session)   {usd(per_task)}")
print(f"cost per successful task      {usd(per_task / SUCCESS_RATE)}")

thresh = by_session.usd.mean() + 2 * by_session.usd.std()
outliers = by_session[by_session.usd > thresh]
print(f"\nSessions costing more than mean + 2 standard deviations ({usd(thresh)}):")
show(outliers.style.format({"usd": "${:,.6f}"})) if len(outliers) else print("  none")


,calls,usd,tokens,use_case
session_id,,,,
6b3b2b69,6,$0.002180,1425,uc-999
018e926a,1,$0.000421,21,nan
e8babf9e,1,$0.000263,23,uc-301
8c8fc08c,1,$0.000229,29,uc-101
d07a986c,1,$0.000225,20,uc-301



cost per call                 $0.000194
cost per task (per session)   $0.000245
cost per successful task      $0.000266

Sessions costing more than mean + 2 standard deviations ($0.001205):


,calls,usd,tokens,use_case
session_id,,,,
6b3b2b69,6,$0.002180,1425,uc-999


---
## 7. Export

Finally we write the spans to `spans.jsonl`, one JSON object per line. In a real deployment you'd send them to an OpenTelemetry collector or straight to an LLM observability tool such as Langfuse or Helicone, and the budget check would move into a gateway.


In [9]:
out = Path("spans.jsonl")
with out.open("w") as f:
    for s in TRACES:
        f.write(json.dumps(s) + "\n")
print(f"{len(TRACES)} spans written to {out} (next to this notebook)")


24 spans written to spans.jsonl (next to this notebook)


If you move budgets into LiteLLM, a few details are worth knowing. It can enforce budgets at several levels: the whole proxy, a team, an internal user, a virtual key, a model, an end customer, or a tag. Each budget can have a reset period (`budget_duration`, for example `30d` or `1h`). **If you don't set one, the budget never resets**, so a monthly limit silently becomes a lifetime limit and everything stops on day 31. Set the reset period explicitly and test it.


In [10]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.004646


,label,model,input,output,cache_write,cache_read,usd,note
0,customer-ops/uc-101,claude-haiku-4-5,30,34,0,0,0.000200,
1,customer-ops/uc-101,claude-haiku-4-5,29,40,0,0,0.000229,
2,customer-ops/uc-102,claude-haiku-4-5,26,5,0,0,0.000051,
3,customer-ops/uc-102,claude-haiku-4-5,22,5,0,0,0.000047,
4,engineering/uc-201,claude-haiku-4-5,34,34,0,0,0.000204,
5,engineering/uc-201,claude-haiku-4-5,21,21,0,0,0.000126,
6,sales-mktg/uc-301,claude-haiku-4-5,20,41,0,0,0.000225,
7,sales-mktg/uc-301,claude-haiku-4-5,23,48,0,0,0.000263,
8,sales-mktg/uc-302,claude-haiku-4-5,20,35,0,0,0.000195,
9,sales-mktg/uc-302,claude-haiku-4-5,20,27,0,0,0.000155,


---
## What to take away

- Work in the order Track → Attribute → Control → Optimize. Without the first three you can't prove or keep the savings from the fourth.
- Tag every call from day one, and enforce it in the wrapper or gateway. Tags can't be added after the fact.
- Test that budgets actually fire, and give each team its own budget key so one overrun doesn't block everyone.
- Give agents several independent per-run limits: steps, tokens, time, and money.
- Report cost per successful task, grouped by session, alongside quality.


### Check yourself

**1. Your dashboard shows 18% of LLM spend as 'unallocated'. What's the most effective fix?**

<details><summary>Show answer</summary>

Make the attribution tags mandatory at the point every call passes through (a shared client wrapper or the gateway): reject or quarantine untagged calls, and give owners a deadline. Chasing individual teams to remember to tag doesn't scale, and the untagged history can't be recovered.

</details>

**2. A team's monthly budget is \$1,000. Spend is \$999.98 and the next request could cost up to \$0.40. Does the budget check stop it? What limits the overshoot?**

<details><summary>Show answer</summary>

No. The check compares spend *so far* (\$999.98) with the limit, so the request is allowed and spend ends at up to \$1,000.38. The overshoot is bounded by the most one call can cost, which is why `max_tokens` (and the input size) should be bounded too.

</details>

**3. An agent calls a slow external API that hangs for ten minutes on each step, but each step uses very few tokens. Which breaker catches it?**

<details><summary>Show answer</summary>

The wall-clock limit. Token and spend limits barely move while the agent is waiting, and the step count grows slowly. That's why you want independent limits rather than just one.

</details>


### Try it on your own work

Add `business_unit`, `use_case_id`, and `session_id` to every LLM call you ship this week, even if nothing reads them yet. Then set a small budget on a non-production key and deliberately exceed it, so you know what the failure looks like before it happens for real.
